# Token Importance / Attention Analysis

## Purpose

Diffusion Planner receives not only ego history but also many tokens representing
neighboring agents, lanes, routes, road boundaries, and other context. This experiment
determines **which inputs the trained model actually uses**.

It addresses four questions:

1. How much does prediction accuracy degrade when neighbors or lanes are removed?
2. Are all 320 neighbor and 140 lane slots necessary, or are nearby tokens sufficient?
3. Which token classes receive attention inside the Fusion Encoder?
4. Are distant tokens useful, or do they behave like noise?

These results help identify candidates for reducing token capacity and inference cost.
They describe the behavior of **this checkpoint on the selected evaluation data**; they
do not prove that an input can be removed safely. Final decisions require retraining and
closed-loop evaluation.

## Running the analysis

`run_token_analysis.sh` runs the analyses and builds both Japanese and English reports.

```bash
MODEL_DIR=best_models/20260730/best_model \
DATADIR=/path/to/dataset \
N_SAMPLES=1024 BATCH_SIZE=64 DEVICE=cuda \
./run_token_analysis.sh
```

The workflow:

1. Run input ablations with `scripts/token_importance.py` and save FDE/ADE to TSV.
2. Aggregate attention and valid-token statistics with `scripts/attention_analysis.py`.
3. Execute this notebook and create a self-contained HTML report.

Plot images and styles are embedded in the HTML, so the report can be viewed on another
machine without the repository, TSV, or JSON files.

## Feature Importance Method (Input Ablation)

This is not tree-model feature importance. It is an **ablation test that measures how
prediction error changes when selected input information is hidden**.

### Procedure

1. Select evaluation scenes whose displacement is at least `MOVE_MIN_M`.
2. Run the unmodified input and compute baseline FDE/ADE.
3. Run the same scenes again after replacing only the target input with zeros.
4. Record `ablation error - baseline error` as importance (delta).

The same scenes, model, and initial trajectory are used, making paired differences easy
to interpret.

### Class-level drop

`drop:neighbors` and `drop:lanes` zero an entire input class.

- Variable-length inputs such as neighbors and maps become padding and are excluded by
  the attention mask.
- Goal pose and turn indicators are fixed tokens. Their information is replaced by a
  constant, but the token itself remains.

### Nearest Top-K

`nbr_top:16` keeps only the 16 neighbors closest to the ego. The smallest K whose error
converges to baseline is a candidate token capacity. Distance is current Euclidean
distance; it does not rank by future collision risk or road connectivity.

### Radius cutoff

`nbr_within:50` keeps neighbors within 50 m. Top-K studies count limits, whereas radius
cutoffs study the physical input range.

### Interpreting the delta

- **Delta > 0:** removing the input hurts accuracy; the checkpoint likely uses it.
- **Delta near 0:** no clear contribution is visible for this data and metric.
- **Delta < 0:** removal improves accuracy; possible causes include noisy/misused input,
  sampling variation, or distribution shift.

A near-zero delta does not prove that an input is unnecessary. Effects on rare
safety-critical scenes may disappear in mean FDE/ADE, and zero-filled input can be
out-of-distribution. Confirm any reduction with more samples, closed-loop safety metrics,
and retraining with the reduced input.

## Metrics and Runtime Parameters

### FDE / ADE

- **FDE (Final Displacement Error):** distance between predicted and ground-truth final
  positions. Lower is better.
- **ADE (Average Displacement Error):** mean position error over all predicted timesteps.
  Lower is better.
- **Delta FDE / ADE:** ablation result minus baseline. Positive means degradation.

### Main environment variables

| Variable | Default | Meaning |
|---|---:|---|
| `MODEL_DIR` | `best_models/20260730/best_model` | Directory containing `args.json` and `best_model.pth` |
| `DATADIR` | mini dataset | Evaluation dataset root |
| `VALID_LIST` | `$DATADIR/path_list_valid.json` | List of evaluation NPZ files |
| `N_SAMPLES` | 128 | Maximum number of moving scenes |
| `BATCH_SIZE` | 32 | Inference batch size |
| `DEVICE` | `cuda` | `cuda` or `cpu` |
| `MOVE_MIN_M` | 5.0 | Minimum endpoint displacement for scene selection |
| `TURN_DEG` | 15.0 | Endpoint bearing threshold for the turning subset |
| `OUT_DIR` | generated from dataset name | Output directory |

Start with `N_SAMPLES=128` for a smoke test, then use 512-1024 or more for formal
evaluation. Use the same data list and thresholds when comparing runs.

In [ ]:
import os
from pathlib import Path

# Read result paths from the shell; use defaults for interactive execution.
IMPORTANCE_TSV = Path(
    os.environ.get(
        "TOKEN_IMPORTANCE_TSV", "../best_models/20260730/eval/token_importance_n1024.tsv"
    )
)
ATTENTION_JSON = Path(
    os.environ.get(
        "TOKEN_ATTENTION_JSON", "../best_models/20260730/eval/attention_analysis_n1024.json"
    )
)
RUN_PARAMETERS = {
    "model": os.environ.get("TOKEN_REPORT_MODEL_DIR", "interactive/default"),
    "valid_list": os.environ.get("TOKEN_REPORT_VALID_LIST", "interactive/default"),
    "n_samples": os.environ.get("TOKEN_REPORT_N_SAMPLES", "unknown"),
    "batch_size": os.environ.get("TOKEN_REPORT_BATCH_SIZE", "unknown"),
    "device": os.environ.get("TOKEN_REPORT_DEVICE", "unknown"),
    "move_min_m": os.environ.get("TOKEN_REPORT_MOVE_MIN_M", "unknown"),
    "turn_deg": os.environ.get("TOKEN_REPORT_TURN_DEG", "unknown"),
}

print("run parameters:")
for key, value in RUN_PARAMETERS.items():
    print(f"  {key:>11}: {value}")
print()
print(
    "importance:",
    IMPORTANCE_TSV,
    "->",
    "OK" if IMPORTANCE_TSV.exists() else "NOT FOUND (set the result path)",
)
print(
    "attention :", ATTENTION_JSON, "->", "OK" if ATTENTION_JSON.exists() else "NOT FOUND (optional)"
)

In [ ]:
import csv
import json

import matplotlib.pyplot as plt
import numpy as np

# Fixed colors for token classes across all plots.
CAT = {"neighbors": "#2a78d6", "lanes": "#eb6834", "line_strings": "#1baf7a", "route": "#eda100"}
C_WORSE = "#eb6834"  # degradation (+delta)
C_BETTER = "#2a78d6"  # improvement (-delta)
INK = "#1a1a19"
MUTED = "#6b6a63"
GRID = "#e5e4df"


def style(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.title.set_color(INK)
    return ax


rows = None
if IMPORTANCE_TSV.exists():
    with open(IMPORTANCE_TSV) as f:
        rows = [
            {k: (v if k == "config" else float(v)) for k, v in r.items()}
            for r in csv.DictReader(f, delimiter="\t")
        ]
    base = next(r for r in rows if r["config"] == "baseline")
    print(
        f"{len(rows)} configs, baseline fde_top={base['fde_top']:.2f}m ade_top={base['ade_top']:.2f}m"
    )

### Complete Feature-Importance Results

This table shows every configuration and every metric stored in the TSV. `top` evaluates
the trajectory selected by the model; `min` evaluates the candidate closest to ground
truth. They are identical for single-trajectory output. Each delta is relative to the
baseline value of the same metric.

In [ ]:
if rows:
    print(
        f"{'config':<22} {'FDE':>7} {'ΔFDE':>7} {'ADE':>7} {'ΔADE':>7} "
        f"{'minFDE':>8} {'ΔminFDE':>9} {'minADE':>8} {'ΔminADE':>9}"
    )
    for r in rows:
        print(
            f"{r['config']:<22} {r['fde_top']:>7.2f} {r['d_fde_top']:>+7.2f} "
            f"{r['ade_top']:>7.2f} {r['d_ade_top']:>+7.2f} "
            f"{r['min_fde']:>8.2f} {r['d_min_fde']:>+9.2f} "
            f"{r['min_ade']:>8.2f} {r['d_min_ade']:>+9.2f}"
        )
    top_equals_min = all(
        np.isclose(r["fde_top"], r["min_fde"]) and np.isclose(r["ade_top"], r["min_ade"])
        for r in rows
    )
    if top_equals_min:
        print(
            "\nNote: top and min are identical for every configuration (single-trajectory output)."
        )

## 1. Feature Importance by Input Class

The two panels show delta FDE and delta ADE after hiding an entire input class. FDE
measures the final point, whereas ADE measures the whole path.

- A longer positive bar means stronger degradation and likely stronger dependency.
- A value near zero means no clear effect was observed for this data and metric.
- A negative value means removal improved the metric, but is not sufficient evidence to
  delete the input.
- A large delta FDE only suggests endpoint impact; a large delta ADE only suggests impact
  on the intermediate path.

Compare both sign and magnitude, and check whether small differences reproduce across
sample sizes and splits. Goal pose and turn indicators are constant-value ablations, not
complete token removal.

In [ ]:
if rows:
    drops = [r for r in rows if r["config"].startswith("drop:")]
    drops.sort(key=lambda r: r["d_fde_top"])
    names = [r["config"][5:] for r in drops]

    fig, axes = plt.subplots(1, 2, figsize=(12, 0.45 * len(drops) + 1.3), sharey=True)
    for ax, metric, label in (
        (axes[0], "d_fde_top", "FDE"),
        (axes[1], "d_ade_top", "ADE"),
    ):
        vals = [r[metric] for r in drops]
        colors = [C_WORSE if v > 0 else C_BETTER for v in vals]
        ax.barh(names, vals, color=colors, height=0.62)
        ax.axvline(0, color=MUTED, lw=1)
        for i, v in enumerate(vals):
            ax.text(
                v + (0.03 if v >= 0 else -0.03),
                i,
                f"{v:+.2f}",
                va="center",
                ha="left" if v >= 0 else "right",
                fontsize=9,
                color=INK,
            )
        ax.set_xlabel(f"Δ top-mode {label} [m]", color=MUTED)
        ax.set_title(f"{label} importance (baseline {base[label.lower() + '_top']:.2f} m)")
        ax.grid(axis="x", color=GRID, lw=0.6)
        ax.set_axisbelow(True)
        style(ax)
    plt.tight_layout()
    plt.show()

## 2. Nearest Top-K and Radius Cutoffs

K is the number of retained tokens. The top row shows FDE, the bottom row shows ADE, and
dashed lines show the all-token baseline.

### Reading the curves

- Error falls as K grows: additional tokens provide useful information.
- Both FDE and ADE stabilize near baseline: that K is a capacity candidate.
- Error grows as K grows: distant tokens may add noise.
- FDE and ADE converge at different K values: endpoint and path quality may require
  different capacities.
- Irregular curves may indicate insufficient samples, unsuitable distance ranking, or
  out-of-distribution ablations.

Define acceptable delta FDE and delta ADE before choosing the smallest K. Top-K and radius
cutoffs answer different design questions and need not give the same conclusion.

In [ ]:
if rows:
    sweeps = {
        "nbr_top": ("neighbors", 320),
        "lane_top": ("lanes", 140),
        "ls_top": ("line_strings", 60),
    }
    metrics = [("fde_top", "FDE"), ("ade_top", "ADE")]
    fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharey=False)
    for ri, (metric, metric_label) in enumerate(metrics):
        for ax, (prefix, (cls, slots)) in zip(axes[ri], sweeps.items()):
            pts = sorted(
                (
                    (int(r["config"].split(":")[1]), r[metric])
                    for r in rows
                    if r["config"].startswith(prefix + ":")
                ),
            )
            ks = [k for k, _ in pts] + [slots]
            ys = [y for _, y in pts] + [base[metric]]
            ax.plot(ks, ys, marker="o", ms=5, lw=2, color=CAT[cls])
            ax.axhline(base[metric], color=MUTED, lw=1, ls="--")
            ax.text(ks[0], base[metric], " baseline", fontsize=8, color=MUTED, va="bottom")
            ax.set_xscale("log", base=2)
            ax.set_xticks(ks[:-1] + [slots])
            ax.set_xticklabels([str(k) for k in ks[:-1]] + [f"all\n({slots})"], fontsize=8)
            ax.set_title(f"{cls}: keep nearest K", fontsize=10)
            ax.set_xlabel("K (log scale)", color=MUTED)
            ax.grid(axis="y", color=GRID, lw=0.6)
            ax.set_axisbelow(True)
            style(ax)
        axes[ri, 0].set_ylabel(f"top-mode {metric_label} [m]", color=MUTED)
    plt.tight_layout()
    plt.show()

    within = [r for r in rows if "_within:" in r["config"]]
    if within:
        print("Radius cutoffs:")
        for r in within:
            print(
                f"  {r['config']:<16} "
                f"FDE={r['fde_top']:6.2f} (Δ={r['d_fde_top']:+.2f})  "
                f"ADE={r['ade_top']:6.2f} (Δ={r['d_ade_top']:+.2f})"
            )

## 3. Attention Analysis and Interpretation

### What attention represents

In the Fusion Encoder, each token (query) assigns weights to other tokens (keys). Weights
over valid keys sum to one for each query. This report averages weights across heads.

Attention shows where the model reads information; it does not directly measure how many
meters that information changes the final prediction. Interpret it together with
ablation importance.

### Reported quantities

- **count share:** fraction of all valid tokens belonging to the class
- **ego-query share:** attention sent from the ego token to the class
- **all-query share:** mean attention received from all valid queries
- **selectivity:** `attention share / count share`
- **value-weighted share:** attention multiplied by value-vector magnitude; an
  approximation that omits per-head structure and output projection

### Selectivity

- Near **1.0:** attention is roughly proportional to token count.
- Above **1.0:** the class is preferred beyond its count share.
- Below **1.0:** the class receives less attention than its count share.

Always inspect both selectivity and absolute share. A rare class can have high
selectivity but little total attention; a numerous class can dominate total attention
despite selectivity below one.

### Layers, distance, and turning subsets

- A class whose share increases in deeper layers may be integrated later.
- Distance bins show the within-class attention distribution and include only scenes
  where that class is present.
- Distance-bin values are not normalized by token count per bin.
- Turning/straight route share is descriptive; the smaller subset can be unstable.

### Combining attention and ablation

| Attention | Ablation degradation | Candidate interpretation |
|---|---|---|
| High | Large | Strongly read and influential |
| High | Small | Read but redundant, or weak effect on output |
| Low | Large | Sparse but important information, or effect through another layer/query |
| Low | Small | Limited use under this checkpoint and evaluation |

These are diagnostic hypotheses, not causal proof. Combine reproducibility, Top-K,
radius cutoffs, and closed-loop safety evaluation before reducing inputs.

In [ ]:
attn = None
if ATTENTION_JSON.exists():
    attn = json.loads(ATTENTION_JSON.read_text())
    print(f"n_samples={attn['n_samples']} (turning {attn['n_turning']}), layers={attn['n_layers']}")

if attn:
    classes = [
        c for c in attn["classes"] if attn["count_share"].get(c, 0) > 0 and c not in ("static",)
    ]
    sel = {c: attn["all_share_avg"][c] / attn["count_share"][c] for c in classes}
    print(
        f"{'class':<16}{'count':>9}{'ego avg':>11}{'all avg':>11}"
        f"{'value-wtd':>12}{'selectivity':>13}"
    )
    for c in attn["classes"]:
        count = attn["count_share"][c]
        ego_avg = float(np.mean(attn["ego_share_per_layer"][c]))
        all_avg = attn["all_share_avg"][c]
        value_avg = attn["vw_share_avg"][c]
        selectivity = all_avg / count if count > 0 else float("nan")
        print(
            f"{c:<16}{100 * count:>8.2f}%{100 * ego_avg:>10.2f}%"
            f"{100 * all_avg:>10.2f}%{100 * value_avg:>11.2f}%{selectivity:>12.2f}x"
        )
    print()
    order = sorted(classes, key=lambda c: sel[c])

    fig, ax = plt.subplots(figsize=(7, 0.45 * len(order) + 1))
    vals = [sel[c] for c in order]
    colors = [C_WORSE if v > 1 else C_BETTER for v in vals]
    ax.barh(order, vals, color=colors, height=0.62)
    ax.axvline(1.0, color=MUTED, lw=1)
    ax.text(1.0, len(order) - 0.2, " 1.0 = count-proportional (dilution)", fontsize=8, color=MUTED)
    for i, v in enumerate(vals):
        ax.text(v + 0.05, i, f"{v:.2f}x", va="center", fontsize=9, color=INK)
    ax.set_xlabel("selectivity (all-query attention share / token count share)", color=MUTED)
    ax.set_title("Attention selectivity by token class")
    ax.grid(axis="x", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()

In [ ]:
if attn:
    layer_labels = [f"L{i}" for i in range(attn["n_layers"])]
    print(f"{'class':<16}" + "".join(f"{label:>9}" for label in layer_labels))
    for c in attn["classes"]:
        layer_values = attn["ego_share_per_layer"][c]
        print(f"{c:<16}" + "".join(f"{100 * v:>8.2f}%" for v in layer_values))
    print()
    key_classes = [
        c for c in ("neighbors", "lanes", "line_strings", "route") if c in attn["classes"]
    ]
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    L = attn["n_layers"]
    for c in key_classes:
        ys = [100 * v for v in attn["ego_share_per_layer"][c]]
        ax.plot(range(L), ys, marker="o", ms=4, lw=2, color=CAT[c])
        ax.text(L - 1 + 0.08, ys[-1], c, fontsize=9, color=CAT[c], va="center")
    ax.set_xticks(range(L))
    ax.set_xticklabels([f"L{i}" for i in range(L)])
    ax.set_xlim(-0.3, L + 1.3)
    ax.set_xlabel("fusion layer", color=MUTED)
    ax.set_ylabel("ego-query attention share [%]", color=MUTED)
    ax.set_title("Ego-token attention share per fusion layer")
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()

In [ ]:
if attn:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))

    ax = axes[0]
    route_groups = [
        ("turning", attn["route_share_turning"]),
        ("straight", attn["route_share_straight"]),
    ]
    route_groups = [(label, value) for label, value in route_groups if value is not None]
    labels = [label for label, _ in route_groups]
    vals = [100 * value for _, value in route_groups]
    print("route share:", "  ".join(f"{label}={value:.2f}%" for label, value in zip(labels, vals)))
    ax.bar(labels, vals, color=CAT["route"], width=0.5)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.15, f"{v:.1f}%", ha="center", fontsize=10, color=INK)
    ax.set_ylabel("route attention share [%]", color=MUTED)
    ax.set_title("Route share: turning vs straight (ego query)")
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)

    ax = axes[1]
    bins = attn["dist_bins"]
    print("distance-bin share:")
    for label, lane_value, nbr_value in zip(bins, attn["lane_bin_share"], attn["nbr_bin_share"]):
        print(f"  {label:<10} lanes={100 * lane_value:6.2f}%  neighbors={100 * nbr_value:6.2f}%")
    x = np.arange(len(bins))
    w = 0.38
    ax.bar(
        x - w / 2, [100 * v for v in attn["lane_bin_share"]], w, color=CAT["lanes"], label="lanes"
    )
    ax.bar(
        x + w / 2,
        [100 * v for v in attn["nbr_bin_share"]],
        w,
        color=CAT["neighbors"],
        label="neighbors",
    )
    ax.set_xticks(x)
    ax.set_xticklabels(bins, fontsize=9)
    ax.set_ylabel("within-class attention share [%]", color=MUTED)
    ax.set_title("Attention by distance bin (ego query)")
    ax.legend(frameon=False, fontsize=9)
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()

## 4. Valid Token Counts and Slot Utilization

Valid-token counts are computed from the padding mask during attention analysis and saved
under `token_occupancy` in the attention JSON. They use the same selected moving scenes;
no separate dataset scan is required.

- **slots:** maximum allocated model slots
- **mean:** mean valid count per scene
- **p50:** half of scenes have this count or fewer
- **p95 / p99:** 95% / 99% of scenes have this count or fewer
- **max:** largest count observed in this run
- **mean utilization:** `mean / slots * 100`

p95 or p99 can seed a capacity proposal. `p99_capacity` is the smallest integer that
covers 99% of this sample, but dropping the remaining 1% is not automatically safe.
Combine occupancy with Top-K behavior, the truncation rule, and closed-loop evaluation.

Utilization measures occupancy, not usefulness. Rare tokens may be important, while a
densely occupied class may still be redundant.

In [ ]:
occupancy = attn.get("token_occupancy") if attn else None
if occupancy:
    by_class = occupancy["by_class"]
    names = list(by_class)
    print(
        f"{'class':<16}{'slots':>7}{'mean':>9}{'p50':>7}{'p95':>7}"
        f"{'p99':>7}{'max':>7}{'mean util':>12}"
    )
    for name in names:
        s = by_class[name]
        print(
            f"{name:<16}{s['slots']:>7}{s['mean']:>9.1f}{s['p50']:>7.1f}"
            f"{s['p95']:>7.1f}{s['p99']:>7.1f}{s['max']:>7}"
            f"{s['mean_utilization_pct']:>11.1f}%"
        )
    total = occupancy["total"]
    print(
        f"\ntotal: mean={total['mean']:.1f}/{total['slots']} "
        f"({total['mean_utilization_pct']:.1f}%), "
        f"p95={total['p95']:.1f}, p99={total['p99']:.1f}, max={total['max']}"
    )

    variable_names = [n for n in names if by_class[n]["slots"] > 1]
    x = np.arange(len(variable_names))
    width = 0.2
    fig, ax = plt.subplots(figsize=(9, 4.2))
    series = [
        ("mean", "mean", "#2a78d6"),
        ("p95", "p95", "#1baf7a"),
        ("p99", "p99", "#eda100"),
        ("max", "max", "#eb6834"),
    ]
    for i, (label, key, color) in enumerate(series):
        values = [100 * by_class[n][key] / by_class[n]["slots"] for n in variable_names]
        ax.bar(x + (i - 1.5) * width, values, width, label=label, color=color)
    ax.set_xticks(x)
    ax.set_xticklabels(variable_names, rotation=20, ha="right")
    ax.set_ylabel("slot utilization [%]", color=MUTED)
    ax.set_title("Valid-token occupancy by class")
    ax.axhline(100, color=MUTED, lw=1, ls="--")
    ax.legend(frameon=False, ncol=4)
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()
else:
    print("token_occupancy is not present; rerun attention_analysis.py with the updated script.")

## 5. Interpretation Checklist

1. Are baseline FDE/ADE plausible, and is the sample count sufficient?
2. Are class-drop deltas practically meaningful, not merely nonzero?
3. Do trends reproduce across sample sizes and data splits?
4. Where do Top-K curves stabilize, and do radius-cutoff results agree?
5. Do mean, p95, p99, and maximum counts expose rare dense scenes?
6. Is attention share explained only by token count? Check selectivity.
7. Why do ego-query, all-query, and value-weighted shares differ?
8. Do attention and ablation agree? Avoid forcing a useful/useless conclusion when they do
   not.
9. Have rare safety-critical scenes been checked individually and in closed loop?

This notebook supports conclusions about **input dependency, attention patterns, and token
occupancy for the current checkpoint on the selected data**. Claims about general input
necessity or safety after removal require retraining, multiple splits, and closed-loop
evaluation.